
# **Interactive exploration of Southern Ocean surface CO₂ and hydrographic variability**


*How do surface CO₂, salinity, and temperature vary across the Southern Ocean, and what do these patterns reveal about ocean-atmosphere carbon exchange?*

For an interactive version of this page please visit the Google Colab: [Open in Google Colab](https://colab.research.google.com/drive/1jZM0ka2pJDbrGEx5jG4H3wD5Vcc4bvYA?usp=drive_link#scrollTo=De7-0m8lZU-n)


*(To open link in new tab press Ctrl + click)*

Alternatevly this notebook can be opened with Binder by following the link: [Interactive exploration of Southern Ocean surface CO₂ and hydrographic variability](https://mybinder.org/v2/gh/s4oceanice/literacy.s4oceanice/main?urlpath=%2Fdoc%2Ftree%2Fnotebooks_binder%2FSOCATv2.ipynb)

**Scientific background**

The Southern Ocean plays a major role in the global carbon cycle by exchanging carbon dioxide with the atmosphere and transporting carbon into the ocean interior. Surface-ocean fugacity of carbon dioxide (`fCO₂`) is influenced by temperature, circulation, biological activity, freshwater inputs, and sea-ice processes.

Combining surface `fCO₂` with sea-surface temperature and salinity helps users explore spatial and temporal variability in the environmental conditions controlling ocean-atmosphere carbon exchange.

**Notebook objectives**

This notebook provides an interactive environment to explore surface ocean carbon dioxide fugacity (fCO₂), salinity, and sea surface temperature (SST) across the Southern Ocean, using gridded monthly data served through the ERDDAP service hosted by OCEAN ICE. This notebook enables users to:

- retrieve monthly gridded SOCAT observations from ERDDAP;
- explore the number of contributing cruises;
- visualize surface `fCO₂`, sea-surface temperature, or salinity;
- select an observation year interactively;
- examine spatial data coverage around Antarctica.

**Data source**

The notebook uses the following dataset:https://er1.s4oceanice.eu/erddap/griddap/SOCATv2024_tracks_gridded_monthly

This dataset is hosted by the OCEAN ICE platform via ERDDAP and is derived from the Surface Ocean CO₂ Atlas (SOCAT), a quality-controlled synthesis of surface water fCO₂ observations collected by research vessels, voluntary observing ships, and moorings worldwide. The gridded monthly product aggregates these measurements, together with co-located salinity and sea surface temperature, onto a regular spatial grid south of 40°S.

The queried variables include:

- `count_ncruise`: number of contributing cruises;
- `fco2_ave_weighted`: weighted-average surface-ocean CO₂ fugacity;
- `salinity_ave_weighted`: weighted-average salinity;
- `sst_ave_weighted`: weighted-average sea-surface temperature.

Note: Southern Ocean observations in SOCAT are concentrated in the austral summer months (approximately November–March), reflecting the seasonal accessibility of the region for research vessels. Data shown for a given year therefore represent this observing season rather than a full annual cycle, and should be interpreted accordingly.

**How to use this notebook**

1. Run the code cells sequentially from top to bottom.
2. Wait for the remote dataset to be downloaded and processed.
3. Use the available menus, sliders, or map controls to select the variables and periods of interest.
4. Read the interpretation guidance before drawing scientific conclusions from the visualizations.

The notebook retrieves data from remote services. An active internet connection is therefore required.


**1. Software environment**

The following cell installs and imports the libraries required for data handling, polar mapping, plotting, and interactive controls.


In [ ]:
# @title
!pip install cartopy
import pandas as pd
import plotly.express as px
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import ipywidgets as widgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 30.2 MB/s eta 0:00:00


## **2. Data retrieval and preparation**

The next section builds the ERDDAP request, downloads the monthly gridded observations, removes the units row, converts time and measurement fields, and derives the year used by the interactive selector.

In [ ]:
# @title
# ERDDAP query
url_1990 = "https://er1.s4oceanice.eu/erddap/griddap/SOCATv2024_tracks_gridded_monthly.csv?count_ncruise%5B(1990-01-16T12:00:00Z):1:(2023-12-16T12:00:00Z)%5D%5B(-90):1:(-40)%5D%5B(-179.5):1:(179.5)%5D,fco2_ave_weighted%5B(1990-01-16T12:00:00Z):1:(2023-12-16T12:00:00Z)%5D%5B(-90):1:(-40)%5D%5B(-179.5):1:(179.5)%5D,salinity_ave_weighted%5B(1990-01-16T12:00:00Z):1:(2023-12-16T12:00:00Z)%5D%5B(-90):1:(-40)%5D%5B(-179.5):1:(179.5)%5D,sst_ave_weighted%5B(1990-01-16T12:00:00Z):1:(2023-12-16T12:00:00Z)%5D%5B(-90):1:(-40)%5D%5B(-179.5):1:(179.5)%5D"

print("Caricamento dati dal 1990 (inclusa SST) in corso...")
df = pd.read_csv(url_1990, skiprows=[1])

# Data cleaning
df['time'] = pd.to_datetime(df['time'])
df['year'] = df['time'].dt.year

# Keep rows where at least one variable of interest is available:
# fCO2 (sea surface CO2 fugacity), salinity, or SST (sea surface temperature)
df_clean = df.dropna(subset=['fco2_ave_weighted', 'salinity_ave_weighted', 'sst_ave_weighted'], how='all').copy()


Caricamento dati dal 1990 (inclusa SST) in corso...


 **3. Interactive Antarctic visualization**

The final code section defines the polar map, filters observations by year, applies variable-specific units and colour limits, and connects the map to the interactive variable and year controls.

elect a variable and year using the controls displayed above the map.

The plotted cells represent locations with available SOCAT observations for the selected period. Blank areas do not necessarily indicate low values; they may instead represent an absence of contributing measurements after gridding and filtering.

In [ ]:
# @title
def show_antarctic_map(variable='fco2_ave_weighted', selected_year=2023):
    # Mapping of measurement unit
    units = {
        'fco2_ave_weighted': 'uatm',
        'salinity_ave_weighted': 'PSU',
        'sst_ave_weighted': '°C'
    }

    # Filter by year
    df_plot = df_clean[df_clean['year'] == selected_year]

    if df_plot.empty or df_plot[variable].isna().all():
        print(f"No data available for {variable} in year {selected_year}.")
        return

    fig = plt.figure(figsize=(10, 10))
    ax = plt.axes(projection=ccrs.SouthPolarStereo())
    ax.set_extent([-180, 180, -90, -40], ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor='#dddddd')
    ax.add_feature(cfeature.OCEAN, facecolor='#f9f9f9', alpha=0.3)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)

    # Semi-transparent points to reveal overlapping observations
    scatter = ax.scatter(
        df_plot['longitude'],
        df_plot['latitude'],
        c=df_plot[variable],
        cmap='plasma',
        s=20,
        alpha=0.5, # Semi-transparent to reveal overlaps
        transform=ccrs.PlateCarree()
    )

    # Add measurement units to the colorbar
    plt.colorbar(scatter, label=f"{variable} [{units.get(variable, '')}]", orientation='vertical', shrink=0.7)
    plt.title(f"Antarctic Observations - Year {selected_year}\nVariable: {variable}", pad=20)

    plt.show()

widgets.interact(
    show_antarctic_map,
    variable=[
        ('fCO2 (uatm)', 'fco2_ave_weighted'),
        ('Salinità (PSU)', 'salinity_ave_weighted'),
        ('Temperatura SST (°C)', 'sst_ave_weighted')
    ],
    selected_year=widgets.IntSlider(value=2023, min=1990, max=2023, step=1, description='Year:')
);

interactive(children=(Dropdown(description='variable', options=(('fCO2 (uatm)', 'fco2_ave_weighted'), ('Salini…

**Interpretation guidance**

SOCAT is an observation-based synthesis rather than a spatially complete model product. Data density is uneven because sampling is concentrated along vessel routes and differs among years and seasons.

Comparisons between years should therefore consider changes in observational coverage and the number of contributing cruises. Surface `fCO₂`, temperature, and salinity should also be interpreted in relation to seasonal timing, sea-ice conditions, biological processes, and regional circulation.


**Additional resources and acknowledgement**
The Python libraries that have been used in this notebook are:

*  [pandas](https://pandas.pydata.org/): for table preparation
*  [cartopy](https://scitools.org.uk/cartopy/docs/latest/): for Antarctic polar mapping
*  [matplotlib](https://matplotlib.org/): for Antarctic polar mapping
*  [ipywidgets](https://ipywidgets.readthedocs.io/en/stable/): for interactive controls

This work has received funding from the European Union Horizon Europe project Ocean-Cryosphere Exchanges in ANtarctica: Impacts on Climate and the Earth System (OCEAN ICE) under grant agreement No. 101060452 (https://doi.org/10.3030/101060452). UK partners are funded by UK Research and Innovation (UKRI) under the UK government's Horizon Europe funding guarantee.

<center>
  <div style="display: flex; justify-content: center; align-items: flex-start; gap: 80px;">
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/TO-USE-RGB-for-digital-materials-V.png" height="140" style="margin-top: 50px;"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/UKRI-logo-1.png" height="100"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2023/06/logo-polar-cluster-2.png" height="100"/>
  </div>
</center>